# Station ID Mapping Check

Checks how well station IDs referenced in `raw_data/raw_trips` (`start_station_id`, `end_station_id`) map onto the reference station table in `raw_data/stations.parquet`.

**Important data-format note (discovered while building this notebook):** `stations.parquet` has *two* ID-like columns:
- `station_id`: a UUID-style string (e.g. `0825be94-1f3f-11e7-bf6b-3863bb334450`) — this is the GBFS station feed's canonical ID.
- `short_name`: a short numeric code as string (e.g. `"32324"`) — this matches the numeric IDs used in `raw_trips`.

`raw_trips.start_station_id` / `end_station_id` are numeric (e.g. `31208.0`) and correspond to `stations.short_name`, **not** `stations.station_id` (a direct check found 0 overlap between trip IDs and `station_id` UUIDs, vs. 817/855 overlap with `short_name`). This notebook therefore joins trip IDs against `stations.short_name`, while still surfacing `station_id` alongside results for reference.

Goals:
- Quantify exact-match coverage between trip station IDs and reference station IDs (`short_name`).
- Surface IDs that appear in trips but are missing from the station reference table (and vice versa).
- Report the row-level match rate (what share of trip rows have both endpoints resolvable to a known station ID).
- Compare ID-based coverage against the name-based coverage from [01_station_mapping_check.ipynb](01_station_mapping_check.ipynb) to see whether joining on ID resolves cases that name-matching missed.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
STATIONS_PATH = RAW_DATA / "stations.parquet"
RAW_TRIPS_PATH = RAW_DATA / "raw_trips"

print(f"Project root: {ROOT}")
print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")
print(f"Raw trips path: {RAW_TRIPS_PATH} (exists={RAW_TRIPS_PATH.exists()})")

Project root: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor
Stations path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\stations.parquet (exists=True)
Raw trips path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\raw_trips (exists=True)


## Load data

- `stations`: reference station table (`raw_data/stations.parquet`), columns include `station_id` (UUID), `short_name` (numeric/alphanumeric code — the actual trip-ID join key), `name`, `lat`, `lon`, `capacity`.
- `trips`: trip data (`raw_data/raw_trips/`), read as a full partitioned parquet dataset so any additional `year=/month=` partitions synced later are picked up automatically. Columns include `start_station_id`, `end_station_id`.

**Schema note:** `start_station_id` / `end_station_id` are stored as `int64` in older partitions (2010 through ~2020-05) but switch to a string type from 2021-02 onward, because some later station IDs are alphanumeric (e.g. `MTL-ECO5-03`, an outlier likely representing a special/placeholder station). Reading the raw partitioned dataset with pandas directly fails with `ArrowInvalid: Failed to parse string ... as a scalar of type int64` because pyarrow tries to unify all partitions under one schema inferred from the first fragment. We work around this by opening the dataset via `pyarrow.dataset` with an explicit schema that forces both ID columns to `string`, so every partition is cast consistently on read.

In [2]:
import pyarrow as pa
import pyarrow.dataset as ds

stations = pd.read_parquet(STATIONS_PATH)

# Build the trips dataset with start/end_station_id forced to string, since their
# on-disk type changes across partitions (int64 pre-2021, alphanumeric string from 2021-02 on).
_raw_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive")
_id_string_fields = []
for field in _raw_dataset.schema:
    if field.name in ("start_station_id", "end_station_id"):
        _id_string_fields.append(pa.field(field.name, pa.string()))
    else:
        _id_string_fields.append(field)
_trips_schema = pa.schema(_id_string_fields)

trips_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive", schema=_trips_schema)
trips = trips_dataset.to_table(
    columns=[
        "start_station_id",
        "end_station_id",
        "start_station_name",
        "end_station_name",
        "started_at",
        "ended_at",
        "duration",
        "period",
    ]
).to_pandas()

# Parse timestamps once, up front, for the first/last-appearance profiling done later.
trips["started_at"] = pd.to_datetime(trips["started_at"], errors="coerce")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], errors="coerce")

print(f"stations: {stations.shape[0]:,} rows, columns: {list(stations.columns)}")
print(f"trips:    {trips.shape[0]:,} rows, columns: {list(trips.columns)}")
print(f"Periods covered in trips: {sorted(trips['period'].dropna().unique())}")

print("\nstations dtypes:")
print(stations.dtypes)
print("\ntrips dtypes:")
print(trips.dtypes)

stations.head()

stations: 866 rows, columns: ['station_id', 'short_name', 'name', 'lat', 'lon', 'capacity', 'updated_at']
trips:    55,897,151 rows, columns: ['start_station_id', 'end_station_id', 'start_station_name', 'end_station_name', 'started_at', 'ended_at', 'duration', 'period']
Periods covered in trips: ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-08', '2018-09', '2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12', '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', 

,station_id,short_name,name,lat,lon,capacity,updated_at
0,2070121870129475816,32324,Scotts Crossing Rd & Spring Gate Dr,38.927018,-77.211094,11,2026-09-26T00:50:25
1,0825be94-1f3f-11e7-bf6b-3863bb334450,31072,31st St & S Woodrow St,38.837846,-77.093478,8,2026-09-26T00:50:25
2,2254169244471500614,32367,Banneker Pool,38.922301,-77.023405,14,2026-09-26T00:50:25
3,2113890246552748136,32304,Westwood Center Dr & Sheraton Tysons Dr,38.929129,-77.246523,12,2026-09-26T00:50:25
4,082569e6-1f3f-11e7-bf6b-3863bb334450,31066,Columbia Pike & S Orme St,38.867217,-77.072296,8,2026-09-26T00:50:25


## Verify which stations column is the real join key

`trips.start_station_id` / `end_station_id` are now loaded as clean strings. `stations.station_id` is a UUID string and `stations.short_name` is a numeric/alphanumeric code string. Check overlap against both stations columns to confirm which one is the actual join key before proceeding.

In [3]:
trip_start_ids_str = trips["start_station_id"].dropna()
trip_end_ids_str = trips["end_station_id"].dropna()
trip_ids = set(trip_start_ids_str.unique()) | set(trip_end_ids_str.unique())

station_id_uuids = set(stations["station_id"].dropna().unique())
station_short_names = set(stations["short_name"].dropna().unique())

overlap_with_station_id = trip_ids & station_id_uuids
overlap_with_short_name = trip_ids & station_short_names

print(f"Unique trip station IDs (start | end):                {len(trip_ids):,}")
print(f"Overlap with stations.station_id (UUIDs):             {len(overlap_with_station_id):,}")
print(f"Overlap with stations.short_name (numeric codes):     {len(overlap_with_short_name):,}")
print(
    "\n=> Using 'stations.short_name' as the join key for trip station IDs "
    "(station_id is a UUID from a different ID space and does not correspond to trip IDs)."
)

Unique trip station IDs (start | end):                926
Overlap with stations.station_id (UUIDs):             0
Overlap with stations.short_name (numeric codes):     866

=> Using 'stations.short_name' as the join key for trip station IDs (station_id is a UUID from a different ID space and does not correspond to trip IDs).


## Build the ID universes

Define the comparison sets: reference station IDs (`stations.short_name`) vs. trip station IDs (`start_station_id` ∪ `end_station_id`). Keep `station_names` (from `stations.name`) around too, for the ID/name cross-check later.

In [4]:
station_ids = station_short_names  # reference set, already computed above
station_names = set(stations["name"].dropna().unique())  # for the ID/name cross-check later
start_ids = set(trip_start_ids_str.unique())
end_ids = set(trip_end_ids_str.unique())
# trip_ids already computed above (start_ids | end_ids)

print(f"Unique reference station IDs (stations.short_name): {len(station_ids):,}")
print(f"Unique start_station_id values:                      {len(start_ids):,}")
print(f"Unique end_station_id values:                        {len(end_ids):,}")
print(f"Unique trip station IDs (start | end):                {len(trip_ids):,}")

Unique reference station IDs (stations.short_name): 866
Unique start_station_id values:                      924
Unique end_station_id values:                        926
Unique trip station IDs (start | end):                926


## Exact-match coverage

How many trip station IDs have an exact match in the station reference table (`short_name`), and how many don't? Unlike names, IDs are exact identifiers, so there's no formatting/casing ambiguity to normalize — an ID either matches or it doesn't.

In [5]:
matched_ids = trip_ids & station_ids
trip_only_ids = trip_ids - station_ids  # in trips, missing from stations
station_only_ids = station_ids - trip_ids  # in stations, never used in trips

coverage_rate = len(matched_ids) / len(trip_ids) if trip_ids else float("nan")

print(f"Trip IDs matched exactly to a station:      {len(matched_ids):,}")
print(f"Trip IDs with NO match in stations:         {len(trip_only_ids):,}")
print(f"Station IDs never referenced in trips:      {len(station_only_ids):,}")
print(f"Exact-match coverage rate (trip IDs):       {coverage_rate:.2%}")

Trip IDs matched exactly to a station:      866
Trip IDs with NO match in stations:         60
Station IDs never referenced in trips:      0
Exact-match coverage rate (trip IDs):       93.52%


### Trip station IDs missing from the station reference table

These are `start_station_id` / `end_station_id` values that don't exactly match any `stations.short_name` value — potential data quality issues, retired/renumbered stations, or IDs from a period before the current live station feed.

In [6]:
trip_only_df = pd.DataFrame(sorted(trip_only_ids, key=lambda x: (len(x), x)), columns=["trip_station_id"])
print(f"{len(trip_only_df):,} trip station IDs have no exact match in stations.parquet (short_name)")
trip_only_df.head(50)

60 trip station IDs have no exact match in stations.parquet (short_name)


,trip_station_id
0,0
1,22907
2,31008
3,31051
4,31095
5,31200
6,31246
7,31248
8,31288
9,31329


### Reference stations never referenced in trips

Shown with `station_id` (UUID) and `name` for readability. These may be legitimate (e.g. stations added after the trip periods currently loaded) rather than data quality issues.

In [7]:
station_only_df = (
    stations.loc[stations["short_name"].isin(station_only_ids), ["short_name", "station_id", "name"]]
    .sort_values("short_name")
    .reset_index(drop=True)
)
print(f"{len(station_only_df):,} reference stations are never referenced in the loaded trip data")
station_only_df.head(50)

0 reference stations are never referenced in the loaded trip data


,short_name,station_id,name


## Cross-check: do unmatched IDs correspond to unmatched names too?

IDs are opaque identifiers, so "closest match" string similarity (used for names in notebook 01) doesn't apply here. Instead, check whether trip rows with an unmatched `start_station_id`/`end_station_id` also have an unmatched `start_station_name`/`end_station_name` — if the ID is bad but the name matches fine (or vice versa), that tells us which field is more reliable for joining.

In [8]:
start_id_matched = trips["start_station_id"].isin(station_ids)
end_id_matched = trips["end_station_id"].isin(station_ids)
start_name_matched = trips["start_station_name"].isin(station_names)
end_name_matched = trips["end_station_name"].isin(station_names)

# Rows where the ID fails to match but the name matches fine, and vice versa.
start_id_bad_name_ok = (~start_id_matched) & start_name_matched
start_name_bad_id_ok = (~start_name_matched) & start_id_matched
end_id_bad_name_ok = (~end_id_matched) & end_name_matched
end_name_bad_id_ok = (~end_name_matched) & end_id_matched

print("Rows where START id is unmatched but START name IS matched: "
      f"{start_id_bad_name_ok.sum():,} ({start_id_bad_name_ok.mean():.2%})")
print("Rows where START name is unmatched but START id IS matched: "
      f"{start_name_bad_id_ok.sum():,} ({start_name_bad_id_ok.mean():.2%})")
print("Rows where END id is unmatched but END name IS matched:     "
      f"{end_id_bad_name_ok.sum():,} ({end_id_bad_name_ok.mean():.2%})")
print("Rows where END name is unmatched but END id IS matched:     "
      f"{end_name_bad_id_ok.sum():,} ({end_name_bad_id_ok.mean():.2%})")

both_id_matched = start_id_matched & end_id_matched
both_name_matched = start_name_matched & end_name_matched
print(f"\nRow-level match rate by ID (both endpoints):   {both_id_matched.mean():.2%}")
print(f"Row-level match rate by NAME (both endpoints): {both_name_matched.mean():.2%}")

Rows where START id is unmatched but START name IS matched: 28,582 (0.05%)
Rows where START name is unmatched but START id IS matched: 4,583,984 (8.20%)
Rows where END id is unmatched but END name IS matched:     38,565 (0.07%)
Rows where END name is unmatched but END id IS matched:     4,681,946 (8.38%)

Row-level match rate by ID (both endpoints):   81.33%
Row-level match rate by NAME (both endpoints): 67.77%


## Row-level match rate (by ID)

Unique-ID coverage doesn't tell us how much of the actual trip *volume* is affected. Compute, per row, whether `start_station_id` and `end_station_id` each resolve to a known station ID, and report the share of rows fully matched on both ends.

In [9]:
total_rows = len(trips)

start_id_match_rate = start_id_matched.mean()
end_id_match_rate = end_id_matched.mean()
both_id_match_rate = both_id_matched.mean()

print(f"Total trip rows:                                       {total_rows:,}")
print(f"Rows with start_station_id matched to a station:       {start_id_matched.sum():,} ({start_id_match_rate:.2%})")
print(f"Rows with end_station_id matched to a station:         {end_id_matched.sum():,} ({end_id_match_rate:.2%})")
print(f"Rows with BOTH endpoints matched (by id):              {both_id_matched.sum():,} ({both_id_match_rate:.2%})")
print(f"Rows with at least one endpoint unmatched (by id):     {(~both_id_matched).sum():,} ({(~both_id_matched).mean():.2%})")

Total trip rows:                                       55,897,151
Rows with start_station_id matched to a station:       49,509,184 (88.57%)
Rows with end_station_id matched to a station:         49,160,527 (87.95%)
Rows with BOTH endpoints matched (by id):              45,463,002 (81.33%)
Rows with at least one endpoint unmatched (by id):     10,434,149 (18.67%)


### Which unmatched IDs account for the most rows?

Ranks unmatched station IDs (start or end) by how many trip rows they appear on, so we can prioritize investigating the ones with the biggest impact. Includes the corresponding station name(s) seen in trips for that ID, for context.

In [10]:
start_id_counts = (
    trips.loc[~start_id_matched, "start_station_id"]
    .value_counts()
    .rename("as_start")
)
end_id_counts = (
    trips.loc[~end_id_matched, "end_station_id"]
    .value_counts()
    .rename("as_end")
)

unmatched_id_impact = (
    pd.concat([start_id_counts, end_id_counts], axis=1)
    .fillna(0)
    .astype(int)
)
unmatched_id_impact["total_rows"] = unmatched_id_impact["as_start"] + unmatched_id_impact["as_end"]

# Attach an example station name seen in trips for each unmatched id, for readability.
name_by_start_id = trips.loc[~start_id_matched].groupby("start_station_id")["start_station_name"].first()
name_by_end_id = trips.loc[~end_id_matched].groupby("end_station_id")["end_station_name"].first()
unmatched_id_impact["example_name"] = name_by_start_id.combine_first(name_by_end_id)

unmatched_id_impact = unmatched_id_impact.sort_values("total_rows", ascending=False)

print(f"{len(unmatched_id_impact):,} distinct unmatched IDs appear across trip rows")
print(f"They account for {unmatched_id_impact['total_rows'].sum():,} row-endpoint occurrences")
unmatched_id_impact.head(60)

60 distinct unmatched IDs appear across trip rows
They account for 4,646,030 row-endpoint occurrences


,as_start,as_end,total_rows,example_name
31200,594503,662217,1256720,Massachusetts Ave & Dupont Circle NW
31613,406826,401342,808168,Eastern Market Metro / Pennsylvania Ave & 7th St SE
31248,394161,407241,801402,Smithsonian-National Mall / Jefferson Dr & 12th St SW
31288,299978,298848,598826,4th St & Madison Dr NW
31246,252855,301950,554805,M St & Pennsylvania Ave NW
31614,92806,105521,198327,11th & H St NE
31642,76497,74722,151219,New Jersey Ave & L St NW
31008,31278,32293,63571,12th & Army Navy Dr
31051,26661,26816,53477,Arlington Blvd & N Queen St
32013,12466,9820,22286,Montgomery & East Ln


## Profile of unmatched IDs: first/last appearance, age, and average duration

For each of the 60 unmatched IDs, compute:
- `first_appearance` / `last_appearance`: earliest/latest trip timestamp involving that ID (as either start or end)
- `age_days`: span between first and last appearance, in days
- `n_rows`: total row-endpoint occurrences (reused from `unmatched_id_impact` above)
- `avg_duration_sec`: mean trip `duration` (seconds) across all occurrences of that ID

This helps separate two different situations: a station with a long `age_days` and high `n_rows` is likely a genuine, long-lived station that's simply missing from the current live snapshot (a good candidate for a manual ID crosswalk entry); a station with a short `age_days` and very low `n_rows` is more likely a short-lived test/placeholder/decommissioned entry that's safe to leave unresolved.

In [11]:
unmatched_ids = unmatched_id_impact.index

# Occurrences of an unmatched id as a trip's start endpoint.
start_occurrences = trips.loc[
    trips["start_station_id"].isin(unmatched_ids),
    ["start_station_id", "started_at", "duration"],
].rename(columns={"start_station_id": "station_id", "started_at": "timestamp"})

# Occurrences of an unmatched id as a trip's end endpoint.
end_occurrences = trips.loc[
    trips["end_station_id"].isin(unmatched_ids),
    ["end_station_id", "ended_at", "duration"],
].rename(columns={"end_station_id": "station_id", "ended_at": "timestamp"})

occurrences = pd.concat([start_occurrences, end_occurrences], ignore_index=True)

profile = occurrences.groupby("station_id").agg(
    first_appearance=("timestamp", "min"),
    last_appearance=("timestamp", "max"),
    avg_duration_sec=("duration", "mean"),
)
profile["age_days"] = (profile["last_appearance"] - profile["first_appearance"]).dt.total_seconds() / 86400
profile["n_rows"] = unmatched_id_impact["total_rows"]
profile["example_name"] = unmatched_id_impact["example_name"]

profile = profile[
    ["example_name", "first_appearance", "last_appearance", "age_days", "n_rows", "avg_duration_sec"]
].sort_values("n_rows", ascending=False)

print(f"Profiled {len(profile):,} unmatched station IDs")
print(f"n_rows total: {profile['n_rows'].sum():,} (should match unmatched_id_impact total: {unmatched_id_impact['total_rows'].sum():,})")
profile

Profiled 60 unmatched station IDs
n_rows total: 4,646,030 (should match unmatched_id_impact total: 4,646,030)


,example_name,first_appearance,last_appearance,age_days,n_rows,avg_duration_sec
station_id,,,,,,
31200,Massachusetts Ave & Dupont Circle NW,2010-09-20 12:30:50,2025-04-01 09:46:55,5306.886169,1256720,972.745679
31613,Eastern Market Metro / Pennsylvania Ave & 7th St SE,2010-10-06 15:03:26,2024-11-19 09:24:41,5157.764757,808168,852.663755
31248,Smithsonian-National Mall / Jefferson Dr & 12th St SW,2012-04-02 08:49:01,2026-06-08 08:14:24,5179.975961,801402,2550.297562
31288,4th St & Madison Dr NW,2016-06-14 08:10:44,2026-06-05 08:27:54,3643.011921,598826,2155.167836
31246,M St & Pennsylvania Ave NW,2011-12-21 12:01:12,2025-06-11 15:44:49,4921.155289,554805,1198.906237
31614,11th & H St NE,2011-12-12 18:11:40,2021-11-30 06:55:59,3640.530775,198327,764.729241
31642,New Jersey Ave & L St NW,2016-08-02 08:42:47,2025-10-14 09:24:45,3360.029144,151219,849.399165
31008,12th & Army Navy Dr,2010-09-21 09:37:21,2019-01-03 09:23:14,3025.990197,63571,1155.617546
31051,Arlington Blvd & N Queen St,2012-09-05 07:00:42,2025-09-24 07:46:44,4767.031968,53477,999.967493


In [13]:
unmatched_ids = unmatched_id_impact.index

# Occurrences of an unmatched id as a trip's end endpoint.
end_occurrences = trips.loc[
    trips["end_station_id"].isin(unmatched_ids),
    ["end_station_id", "ended_at", "duration"],
].rename(columns={"end_station_id": "station_id", "ended_at": "timestamp"})

# Occurrences of an unmatched id as a trip's end endpoint.
end_occurrences = trips.loc[
    trips["end_station_id"].isin(unmatched_ids),
    ["end_station_id", "ended_at", "duration"],
].rename(columns={"end_station_id": "station_id", "ended_at": "timestamp"})

occurrences = pd.concat([end_occurrences, end_occurrences], ignore_index=True)

profile = occurrences.groupby("station_id").agg(
    first_appearance=("timestamp", "min"),
    last_appearance=("timestamp", "max"),
    avg_duration_sec=("duration", "mean"),
)
profile["age_days"] = (profile["last_appearance"] - profile["first_appearance"]).dt.total_seconds() / 86400
profile["n_rows"] = unmatched_id_impact["total_rows"]
profile["example_name"] = unmatched_id_impact["example_name"]

profile = profile[
    ["example_name", "first_appearance", "last_appearance", "age_days", "n_rows", "avg_duration_sec"]
].sort_values("n_rows", ascending=False)

print(f"Profiled {len(profile):,} unmatched station IDs")
print(f"n_rows total: {profile['n_rows'].sum():,} (should match unmatched_id_impact total: {unmatched_id_impact['total_rows'].sum():,})")
profile

Profiled 60 unmatched station IDs
n_rows total: 4,646,030 (should match unmatched_id_impact total: 4,646,030)


,example_name,first_appearance,last_appearance,age_days,n_rows,avg_duration_sec
station_id,,,,,,
31200,Massachusetts Ave & Dupont Circle NW,2010-09-20 12:30:50,2025-04-01 09:31:55,5306.875752,1256720,922.767926
31613,Eastern Market Metro / Pennsylvania Ave & 7th St SE,2010-10-06 15:03:26,2024-11-19 09:24:41,5157.764757,808168,854.077261
31248,Smithsonian-National Mall / Jefferson Dr & 12th St SW,2012-04-02 08:52:04,2026-06-08 07:58:51,5179.963044,801402,2504.658652
31288,4th St & Madison Dr NW,2016-06-14 08:47:03,2026-06-05 08:27:54,3642.986701,598826,2084.665962
31246,M St & Pennsylvania Ave NW,2011-12-21 12:02:27,2025-06-11 15:44:49,4921.154421,554805,1145.989258
31614,11th & H St NE,2011-12-12 18:11:40,2021-11-30 06:55:59,3640.530775,198327,755.979273
31642,New Jersey Ave & L St NW,2016-08-02 08:47:08,2025-10-14 09:06:45,3360.013623,151219,848.345926
31008,12th & Army Navy Dr,2010-09-21 09:38:43,2019-01-03 09:23:14,3025.989248,63571,1190.489456
31051,Arlington Blvd & N Queen St,2012-09-05 17:19:40,2025-09-23 21:59:58,4766.194653,53477,968.752036


## Summary

In [12]:
from IPython.display import Markdown, display

summary = f"""
| Metric | Value |
|---|---|
| Reference station IDs (`stations.short_name`) | {len(station_ids):,} |
| Unique trip station IDs (start ∪ end) | {len(trip_ids):,} |
| Exact-match coverage (unique IDs) | {coverage_rate:.2%} |
| Trip IDs unmatched | {len(trip_only_ids):,} |
| Reference stations never used in trips (by id) | {len(station_only_ids):,} |
| Trip rows | {total_rows:,} |
| Row-level match rate by ID (both endpoints) | {both_id_match_rate:.2%} |
| Row-level match rate by NAME (both endpoints, for comparison) | {both_name_matched.mean():.2%} |

**Note:** `stations.station_id` (UUID) is a different ID space and was found to have 0 overlap with trip station IDs — `stations.short_name` is the correct join key. See [01_station_mapping_check.ipynb](01_station_mapping_check.ipynb) for the name-based version of this check.
"""

display(Markdown(summary))


| Metric | Value |
|---|---|
| Reference station IDs (`stations.short_name`) | 866 |
| Unique trip station IDs (start ∪ end) | 926 |
| Exact-match coverage (unique IDs) | 93.52% |
| Trip IDs unmatched | 60 |
| Reference stations never used in trips (by id) | 0 |
| Trip rows | 55,897,151 |
| Row-level match rate by ID (both endpoints) | 81.33% |
| Row-level match rate by NAME (both endpoints, for comparison) | 67.77% |

**Note:** `stations.station_id` (UUID) is a different ID space and was found to have 0 overlap with trip station IDs — `stations.short_name` is the correct join key. See [01_station_mapping_check.ipynb](01_station_mapping_check.ipynb) for the name-based version of this check.
